# Move-Outs Linear Regression Model

        This notebook fits an ordinary least squares linear regression model where the dependent variable is **MoveOuts**.

        The source workbook has one row per move-out event, so this notebook first aggregates the data to `DevelopmentName + FiscalYear` and then predicts the move-out count for each development-year group.


In [ ]:
import json
        import math
        from pathlib import Path

        import numpy as np
        import pandas as pd
        import matplotlib.pyplot as plt


## Upload the Excel File

        In Google Colab, run the next cell and choose `CIS internship data .xlsx` from your computer. If you are running this somewhere other than Colab, place the file next to this notebook or update `INPUT` manually.


In [ ]:
INPUT = Path("CIS internship data .xlsx")

        try:
            from google.colab import files

            if not INPUT.exists():
                uploaded = files.upload()
                if uploaded:
                    INPUT = Path(next(iter(uploaded.keys())))
        except ImportError:
            pass

        print(f"Using input file: {INPUT.resolve() if INPUT.exists() else INPUT}")
        if not INPUT.exists():
            raise FileNotFoundError("Upload the Excel file or update INPUT to point to it.")


## Helper Functions


In [ ]:
def fiscal_year_start(value):
            if pd.isna(value):
                return np.nan
            text = str(value)
            digits = "".join(ch if ch.isdigit() else " " for ch in text).split()
            return int(digits[0]) if digits else np.nan


        def yes_rate(series):
            return series.astype(str).str.strip().str.lower().eq("yes").mean()


        def share(series, value):
            return series.astype(str).str.strip().str.lower().eq(value.lower()).mean()


## Load and Prepare the Workbook


In [ ]:
def load_data(input_path):
            xl = pd.ExcelFile(input_path)
            frames = []
            for sheet in xl.sheet_names:
                df = pd.read_excel(input_path, sheet_name=sheet)
                df["SourceSheet"] = sheet
                frames.append(df)

            data = pd.concat(frames, ignore_index=True)
            data["FiscalYearStart"] = data["FiscalYear"].apply(fiscal_year_start)

            numeric_cols = [
                "HOHAgeAtMoveOut",
                "HouseholdRentToCharge",
                "HouseholdUnitTenure",
                "HouseholdProgramTenure",
            ]
            for col in numeric_cols:
                data[col] = pd.to_numeric(data[col], errors="coerce")

            return data


        data = load_data(INPUT)
        print(f"Loaded {len(data):,} move-out event rows from {data['SourceSheet'].nunique()} sheets.")
        data.head()


## Aggregate to the Model Grain


In [ ]:
def aggregate(data):
            rows = []
            grouped = data.groupby(["DevelopmentName", "FiscalYear"], dropna=False)

            for (development, fiscal_year), group in grouped:
                rows.append(
                    {
                        "DevelopmentName": development,
                        "FiscalYear": fiscal_year,
                        "FiscalYearStart": fiscal_year_start(fiscal_year),
                        "MoveOuts": len(group),
                        "AvgAgeAtMoveOut": group["HOHAgeAtMoveOut"].mean(),
                        "AvgRentToCharge": group["HouseholdRentToCharge"].mean(),
                        "AvgUnitTenure": group["HouseholdUnitTenure"].mean(),
                        "AvgProgramTenure": group["HouseholdProgramTenure"].mean(),
                        "PctMoveOutTransfer": yes_rate(group["HouseholdMoveOutTransfer"]),
                        "PctMoveInTransfer": yes_rate(group["HouseholdMoveInTransfer"]),
                        "PctFamily": share(group["HouseholdFamilyTypeHud"], "Family"),
                        "PctElderly": share(group["HouseholdFamilyTypeHud"], "Elderly"),
                        "PctDisabled": share(group["HouseholdFamilyTypeHud"], "Disabled"),
                        "PctElderlyDisabled": share(group["HouseholdFamilyTypeHud"], "Elderly/Disabled"),
                        "PctHispanic": share(group["HOHEthnicity"], "Hispanic or Latino"),
                        "PctVoluntaryTermination": share(group["HouseholdMoveOutCategory"], "Voluntary Termination"),
                        "PctTransferCategory": share(group["HouseholdMoveOutCategory"], "Transfer"),
                        "PctEvicted": share(group["HouseholdMoveOutCategory"], "Evicted"),
                    }
                )

            agg = pd.DataFrame(rows)
            return agg.sort_values(["FiscalYearStart", "DevelopmentName"]).reset_index(drop=True)


        model_dataset = aggregate(data)
        print(f"Aggregated to {len(model_dataset):,} development-year rows.")
        model_dataset.head()


## Fit the Linear Regression


In [ ]:
def fit_ols(df, predictors):
            model_df = df[["DevelopmentName", "FiscalYear", "MoveOuts"] + predictors].dropna().copy()

            y = model_df["MoveOuts"].to_numpy(dtype=float)
            X = model_df[predictors].to_numpy(dtype=float)
            X_design = np.column_stack([np.ones(len(X)), X])

            beta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
            fitted = X_design @ beta
            residuals = y - fitted

            n = len(y)
            p = X_design.shape[1]
            sse = float(np.sum(residuals**2))
            sst = float(np.sum((y - y.mean()) ** 2))
            r2 = 1 - sse / sst if sst else np.nan
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p) if n > p and not np.isnan(r2) else np.nan
            rmse = math.sqrt(sse / n)
            mae = float(np.mean(np.abs(residuals)))

            dof = max(n - p, 1)
            sigma2 = sse / dof
            xtx_inv = np.linalg.pinv(X_design.T @ X_design)
            se = np.sqrt(np.diag(xtx_inv) * sigma2)
            tstats = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se != 0)

            coefficients = pd.DataFrame(
                {
                    "Term": ["Intercept"] + predictors,
                    "Coefficient": beta,
                    "StdError": se,
                    "TStatistic": tstats,
                }
            )

            predictions = model_df[["DevelopmentName", "FiscalYear", "MoveOuts"]].copy()
            predictions["PredictedMoveOuts"] = fitted
            predictions["Residual"] = residuals

            metrics = {
                "Observations": n,
                "Predictors": len(predictors),
                "R2": r2,
                "AdjustedR2": adj_r2,
                "RMSE": rmse,
                "MAE": mae,
                "DependentVariable": "MoveOuts, counted by DevelopmentName + FiscalYear",
            }

            return coefficients, predictions, metrics, model_df


In [ ]:
predictors = [
            "FiscalYearStart",
            "AvgAgeAtMoveOut",
            "AvgRentToCharge",
            "AvgUnitTenure",
            "PctMoveOutTransfer",
            "PctMoveInTransfer",
            "PctFamily",
            "PctElderly",
            "PctDisabled",
            "PctHispanic",
            "PctVoluntaryTermination",
            "PctTransferCategory",
            "PctEvicted",
        ]

        coefficients, predictions, metrics, model_df = fit_ols(model_dataset, predictors)
        pd.DataFrame(metrics.items(), columns=["Metric", "Value"])


## Coefficients


In [ ]:
coefficients


## Actual vs. Predicted Move-Outs


In [ ]:
plt.figure(figsize=(8, 6))
        plt.scatter(predictions["MoveOuts"], predictions["PredictedMoveOuts"], alpha=0.65)

        min_value = min(predictions["MoveOuts"].min(), predictions["PredictedMoveOuts"].min())
        max_value = max(predictions["MoveOuts"].max(), predictions["PredictedMoveOuts"].max())
        plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

        plt.title("Actual vs Predicted Move-Outs")
        plt.xlabel("Actual Move-Outs")
        plt.ylabel("Predicted Move-Outs")
        plt.grid(True, alpha=0.25)
        plt.show()


## Largest Residuals


In [ ]:
predictions.assign(AbsResidual=predictions["Residual"].abs()) \
            .sort_values("AbsResidual", ascending=False) \
            .head(20)


## Save Output Files


In [ ]:
OUT = Path("regression_outputs")
        OUT.mkdir(parents=True, exist_ok=True)

        summary = pd.DataFrame(
            [
                ["Source rows", len(data)],
                ["Aggregated rows", len(model_dataset)],
                ["Model rows after dropping missing predictor values", metrics["Observations"]],
                ["Dependent variable", metrics["DependentVariable"]],
                ["R-squared", metrics["R2"]],
                ["Adjusted R-squared", metrics["AdjustedR2"]],
                ["RMSE", metrics["RMSE"]],
                ["MAE", metrics["MAE"]],
            ],
            columns=["Metric", "Value"],
        )

        model_dataset.to_csv(OUT / "model_dataset.csv", index=False)
        coefficients.to_csv(OUT / "coefficients.csv", index=False)
        predictions.to_csv(OUT / "predictions.csv", index=False)
        summary.to_csv(OUT / "summary.csv", index=False)

        with open(OUT / "metadata.json", "w", encoding="utf-8") as f:
            json.dump({"predictors": predictors, **metrics}, f, indent=2)

        print(f"Saved CSV outputs to: {OUT.resolve()}")


In [ ]:
try:
            from google.colab import files

            for filename in [
                "model_dataset.csv",
                "coefficients.csv",
                "predictions.csv",
                "summary.csv",
                "metadata.json",
            ]:
                files.download(str(OUT / filename))
        except ImportError:
            print("Not running in Colab, so download prompts were skipped.")
